# Data_preperation

In [ ]:
##
import os
from pathlib import Path
import subprocess

urls = [
    "https://cf.10xgenomics.com/samples/xenium/3.0.0/Xenium_Prime_Human_Lung_Cancer_FFPE/Xenium_Prime_Human_Lung_Cancer_FFPE_he_image.ome.tif",
    "https://cf.10xgenomics.com/samples/xenium/3.0.0/Xenium_Prime_Human_Lung_Cancer_FFPE/Xenium_Prime_Human_Lung_Cancer_FFPE_he_imagealignment.csv",
    "https://s3-us-west-2.amazonaws.com/10x.files/samples/xenium/3.0.0/Xenium_Prime_Human_Lung_Cancer_FFPE/Xenium_Prime_Human_Lung_Cancer_FFPE_outs.zip",
]


# download the data
for url in urls:
    filename = Path(url).name
    os.makedirs("data", exist_ok=True)
    command = f"curl -o {'data/' + filename} {url}"
    subprocess.run(command, shell=True, check=True)

# ##
# unzip the data
subprocess.run(
    f"unzip -o data/Xenium_Prime_Human_Lung_Cancer_FFPE_outs.zip -d data/",
    shell=True,
    check=True,
)

In [ ]:
from pathlib import Path
import shutil
from spatialdata_io import xenium
import spatialdata as sd

# 현재 작업 디렉토리를 기준으로 경로 설정
path = Path().resolve()

# Xenium_Prime_Human_Lung_io 디렉토리 설정
#path /= "Human_Lung_Cancer/Xenium_Prime_Human_Lung_io"
path /= "xenium_link/Human_Lung_Cancer/Xenium_Prime_Human_Lung_io"
path.mkdir(parents=True, exist_ok=True)  # 디렉토리가 없으면 생성

# 데이터를 읽을 경로와 저장할 경로 설정
path_read = path / "data"  # 첫 번째 코드에서 만든 "data" 폴더
path_write = path / "data.zarr"  # Zarr 데이터가 저장될 경로

## 데이터 파싱
print("parsing the data... ", end="")
sdata = xenium(
    path=str(path_read),
    n_jobs=8,
    cell_boundaries=True,
    nucleus_boundaries=True,
    morphology_focus=True,
    cells_as_circles=True,
)
print("done")

## 데이터 저장
print("writing the data... ", end="")
if path_write.exists():
    shutil.rmtree(path_write)  # 기존 폴더 삭제
sdata.write(path_write)  # 새로 저장
print("done")

## 데이터 읽기 및 확인
sdata = sd.SpatialData.read(str(path_write))
print(sdata)


# Packages

In [1]:
import spatialdata as sd
import matplotlib.pyplot as plt
import spatialdata_plot
import shapely
import numpy as np
import xarray as xr
import skimage.transform
import pandas as pd
from shapely.geometry import Polygon
import geopandas as gpd
from shapely.geometry import Point
import cv2
import os
import time
import multiprocessing as mp
import scanpy as sc
import pandas as pd
from scipy import sparse
from multiprocessing import Pool

# Global variable

In [2]:
num_cores = 32
affine_matrix_path = "./xenium_prime_link/Human_Lung_Cancer/Xenium_Prime_Human_Lung_Cancer_FFPE_he_imagealignment.csv"
#affine_matrix_path = "./xenium_"
pixel_size = 0.2125  # Xenium 파일 기반 Pixel Size 설정
upper_threshold = 200
#upper_threshold = 300
organ ='10x_5k_lung'

# Functions

In [3]:
# 🔹 1. Pixel 변환 함수 (μm → 픽셀)
def convert_um_to_px(um_x, um_y, pixel_size):
    """마이크로미터(μm) 좌표를 픽셀(pixel) 좌표로 변환"""
    px_x = um_x / pixel_size
    px_y = um_y / pixel_size
    return px_x, px_y

In [4]:
# 🔹 2. Affine 변환 적용 함수 (Xenium → H&E)
def apply_affine(x_pixel, y_pixel, affine_matrix):
    """Affine 변환을 사용하여 Xenium 좌표를 H&E 이미지 좌표로 변환"""
    vec = np.array([[x_pixel], [y_pixel], [1]])  # (3,1) 컬럼 벡터 변환
    x_real, y_real, _ = (affine_matrix @ vec).flatten()  # 행렬 곱 후 1D 변환
    return x_real, y_real

In [5]:
# 🔹 3. Affine 변환을 적용한 Polygon 변환 함수
def apply_affine_to_polygon(polygon, affine_matrix):
    """Polygon의 모든 좌표에 Affine 변환 적용"""
    coords = np.array(polygon.exterior.coords)
    transformed_coords = np.array([apply_affine(x, y, np.linalg.inv(affine_matrix)) for x, y in coords])
    return Polygon(transformed_coords)

In [6]:
# 🔹 4. Affine Matrix 로드 함수
def load_affine_matrix(affine_matrix_path):
    """Affine 변환을 위한 행렬 로드"""
    affine_matrix = pd.read_csv(affine_matrix_path, header=None).values
    #print("✅ Affine Matrix Loaded:\n", affine_matrix)
    return affine_matrix

In [7]:
def convert_polyum_to_px(polygon, pixel_size):
    """Polygon 좌표를 μm → 픽셀 변환"""
    coords = np.array(polygon.exterior.coords)  # 다각형 좌표 추출
    coords_px = coords / pixel_size  # μm → 픽셀 변환
    return Polygon(coords_px)

In [8]:
# ✅ 크롭된 polygon 좌표 변환
def adjust_polygon_for_crop(polygon, crop_x_min, crop_y_min):
    coords = np.array(polygon.exterior.coords)
    adjusted_coords = coords - [crop_x_min, crop_y_min]
    return Polygon(adjusted_coords)

In [9]:
def crop_he_image(cell_id, pixel_size, affine_matrix, he_image_array, image_width, image_height, crop_size):
    """
    특정 Cell ID에 대해 H&E 이미지에서 해당 셀 영역을 크롭하여 반환하는 함수.
    
    Parameters:
        - cell_id (str): 처리할 Cell ID
        - pixel_size (float): Xenium의 픽셀 크기 (μm/px)
        - affine_matrix (np.array): Affine 변환 행렬
        - he_image_array (np.array): H&E 이미지 배열 (C, Y, X)
        - image_width (int): H&E 이미지의 너비
        - image_height (int): H&E 이미지의 높이
        - crop_size (int): 크롭할 정사각형 크기 (기본값=50)
    
    Returns:
        - cropped_image (np.array): 크롭된 H&E 이미지 (C, Y_crop, X_crop)
        - crop_x_min, crop_x_max, crop_y_min, crop_y_max (int): 크롭된 좌표
    """

    #print(f"🔍 Cropping H&E Image for Cell ID: {cell_id}")

    # ✅ 1. Polygon 기반으로 H&E 이미지 크롭
    cell_boundaries = sdata.shapes["cell_boundaries"]
    #print(cell_boundaries)
    #if cell_id not in list(cell_boundaries.index):
        #print(f"⚠️ Cell ID {cell_id}를 찾을 수 없습니다.")
        #return None, None, None, None, None

    # ✅ 2. 셀의 Polygon과 중심 좌표 가져오기
    cell_shape = cell_boundaries.loc[cell_id, "geometry"]
    centroid = cell_shape.centroid
    #cell_x, cell_y = int(centroid.x), int(centroid.y)

    # ✅ 3. Xenium Explorer의 μm 좌표를 픽셀 좌표로 변환
    xenium_x_um, xenium_y_um = centroid.x, centroid.y
    xenium_x_px, xenium_y_px = xenium_x_um / pixel_size, xenium_y_um / pixel_size

    #print(f"📏 Xenium μm 좌표: (x={xenium_x_um:.2f}, y={xenium_y_um:.2f})")
    #print(f"📏 Xenium 픽셀 좌표: (x={xenium_x_px:.2f}, y={xenium_y_px:.2f})")

    # ✅ 4. Affine 변환 적용 (Xenium → H&E)
    he_x, he_y = apply_affine(xenium_x_px, xenium_y_px, np.linalg.inv(affine_matrix))

    #print(f"✅ 변환된 H&E 좌표 (픽셀 단위): (x={he_x:.2f}, y={he_y:.2f})")
    
    # ✅ 5. 크롭된 H&E 이미지 가져오기
    #half_crop = crop_size // 2

    # ✅ 크롭 좌표 보정 (이미지 경계를 벗어나지 않도록)
    crop_x_min = max(0, int(he_x - crop_size // 2))
    crop_x_max = min(image_width, int(he_x + crop_size // 2))
    crop_y_min = max(0, int(he_y - crop_size // 2))
    crop_y_max = min(image_height, int(he_y + crop_size // 2))

    #print(f"📏 크롭된 영역: X({crop_x_min}:{crop_x_max}), Y({crop_y_min}:{crop_y_max})")

    # ✅ 크롭된 이미지 추출
    cropped_image = he_image_array[:, crop_y_min:crop_y_max, crop_x_min:crop_x_max]

    return cropped_image, crop_x_min, crop_x_max, crop_y_min, crop_y_max

In [10]:
def get_obb_dimensions(polygon):
    """ 다각형에 외접하는 Rotated Bounding Box(OBB)의 가로, 세로 크기 반환 """
    rotated_bbox = polygon.minimum_rotated_rectangle  # 회전된 최소 직사각형

    # Bounding Box의 꼭짓점 좌표 추출
    coords = np.array(rotated_bbox.exterior.coords[:-1])  # 마지막 점은 첫 점과 동일하므로 제외

    # 너비 및 높이 계산 (가장 긴 두 변을 찾음)
    edge_lengths = np.linalg.norm(coords - np.roll(coords, shift=1, axis=0), axis=1)
    width, height = np.sort(edge_lengths)[-2:]  # 가장 긴 두 변 선택

    return int(width), int(height)  # 정수 변환

In [11]:
def process_polygon(cell_id, pixel_size, affine_matrix, he_image_array, crop_size):
    """ 특정 Cell ID에 대해 Polygon 변환 및 크롭된 H&E 이미지 처리 """

    cell_boundaries = sdata.shapes["cell_boundaries"].copy()

    # ✅ 특정 cell_id에 대한 Polygon 가져오기
    if cell_id not in cell_boundaries.index:
        print(f"⚠️ Cell ID {cell_id}를 찾을 수 없습니다.")
        return
    
    cell_polygon = cell_boundaries.loc[cell_id, "geometry"]

    # ✅ 1. 해당 Polygon을 픽셀 단위로 변환
    cell_polygon_px = convert_polyum_to_px(cell_polygon, pixel_size)
    #print("✅ 선택된 Cell의 좌표가 픽셀 단위로 변환되었습니다.")

    # ✅ 2. Affine 변환 적용
    cell_polygon_transformed = apply_affine_to_polygon(cell_polygon_px, affine_matrix)
    #print("✅ 선택된 Cell의 좌표에 Affine 변환이 적용되었습니다.")

    # ✅ 결과를 원래 데이터프레임에 반영
    cell_boundaries.at[cell_id, "geometry"] = cell_polygon_transformed

    # ✅ 3. 크롭된 이미지 정보 가져오기
    cropped_image, crop_x_min, crop_x_max, crop_y_min, crop_y_max = crop_he_image(
        cell_id, pixel_size, affine_matrix, he_image_array, image_width, image_height, crop_size
    )

    if cropped_image is None:
        print(f"⚠️ Cell ID {cell_id}의 H&E 이미지를 크롭할 수 없음.")
        return
    
    # ✅ 4. Polygon 좌표를 크롭된 이미지 기준으로 변환
    adjusted_polygon = adjust_polygon_for_crop(cell_polygon_transformed, crop_x_min, crop_y_min)

    # ✅ 5. Polygon 내부의 픽셀 좌표 찾기
    polygon_mask = np.zeros((crop_size, crop_size), dtype=np.uint8)
    x, y = np.meshgrid(np.arange(crop_size), np.arange(crop_size))
    points = np.vstack((x.ravel(), y.ravel())).T  # 모든 좌표 조합 생성
    
    for point in points:
        if adjusted_polygon.contains(Point(point[0], point[1])):  # Polygon 내부 확인
            polygon_mask[point[1], point[0]] = 1  # 마스크 적용
    
    # ✅ 6. Polygon 내부만 남기고 나머지는 0으로 처리
    cropped_image_masked = cropped_image.copy()
    cropped_image_masked[:, polygon_mask == 0] = 0  # Polygon 바깥 픽셀 0으로 변환

    # ✅ 7. Annotation 정보 가져오기
    cell_annotation_info = annotation_df[annotation_df["cell_id"] == cell_id]
    if cell_annotation_info.empty:
        print(f"⚠️ Cell ID {cell_id}의 Annotation 정보를 찾을 수 없음.")
        return

    # ✅ 8. 저장할 폴더 경로 설정
    save_path = f"/data0/xenium/paired_dataset/{organ}/he{upper_threshold}/"
    os.makedirs(save_path, exist_ok=True)  # 폴더가 없으면 생성

    # ✅ 9. Polygon이 적용된 이미지 저장 (시간 측정 포함)
    image_filename = f"{save_path}cell_{cell_id}_polygon_HE.png"
    image_filename2 = f"{save_path}cell_{cell_id}_HE.png"

    #print(f"📌 저장 파일: {image_filename}, {image_filename2}")

    # ✅ 이미지 생성 및 저장 시간 측정 시작
    #start_time = time.time()

    # ✅ 10. Polygon 적용된 크롭 이미지 저장
    #fig, ax = plt.subplots(figsize=(5, 5))
    #ax.imshow(cropped_image.transpose(1, 2, 0))  # (C, Y, X) → (Y, X, C)
    
    # ✅ Polygon 추가
    #x, y = adjusted_polygon.exterior.xy
    #ax.plot(x, y, color="red", linewidth=2, label=f"Cell {cell_id}")
    
    # ✅ 중심점 추가
    #ax.scatter([crop_size // 2], [crop_size // 2], color="blue", marker="o", label="Crop Center")
    
    #ax.legend()
    #ax.set_title(f"Cropped H&E Image with Cell {cell_id} Polygon")
    #ax.axis("off")  # 축 제거
    
    #fig_creation_time = time.time()  # ✅ 이미지 생성 완료 시점

    # ✅ 이미지 저장
    #fig.savefig(image_filename, bbox_inches="tight", pad_inches=0, dpi=300)
    #fig_save_time = time.time()  # ✅ 이미지 저장 완료 시점
    #plt.close(fig)

    #print(f"✅ Polygon Applied H&E Image saved: {image_filename}")
    #print(f"📏 이미지 생성 시간: {fig_creation_time - start_time:.4f}초")
    #print(f"💾 이미지 저장 시간: {fig_save_time - fig_creation_time:.4f}초")

    # ✅ 11. Masked 이미지 저장 (시간 측정 포함)
    #start_time2 = time.time()

    #fig, ax = plt.subplots(figsize=(5, 5))
    #ax.imshow(cropped_image_masked.transpose(1, 2, 0))  # (C, Y, X) → (Y, X, C)
    #ax.set_title(f"Corrected H&E Image (Polygon Applied)")
    #ax.axis("off")

    #fig_creation_time2 = time.time()  # ✅ 이미지 생성 완료 시점

    #fig.savefig(image_filename2, bbox_inches="tight", pad_inches=0, dpi=300)
    #fig_save_time2 = time.time()  # ✅ 이미지 저장 완료 시점
    #plt.close(fig)

    #print(f"✅ Masked H&E Image saved: {image_filename2}")
    #print(f"📏 Masked 이미지 생성 시간: {fig_creation_time2 - start_time2:.4f}초")
    #print(f"💾 Masked 이미지 저장 시간: {fig_save_time2 - fig_creation_time2:.4f}초")


In [12]:
# 🔹 5. Cell 데이터 처리 함수
def process_cell(cell_id):
    """ 특정 Cell ID에 대한 정보를 저장하는 함수 """
    #print(f"🔍 Processing cell: {cell_id}")
    
    # ✅ 1. 크롭된 이미지 가져오기
    cropped_image, crop_x_min, crop_x_max, crop_y_min, crop_y_max = crop_he_image(
        cell_id, pixel_size, affine_matrix, he_image_array, image_width, image_height, crop_size
    )

    if cropped_image is None:
        print(f"⚠️ Cell ID {cell_id}의 H&E 이미지를 크롭할 수 없음.")
        return
        
    # ✅ 2. 특정 셀의 Transcript 정보 가져오기
    transcript_df = transcript_data.obs[transcript_data.obs["cell_id"] == cell_id].copy()
    transcript_df["cell_id"] = cell_id  # Cell ID 추가
    transcript_df["cell_id"] = transcript_df["cell_id"].astype(str)  # ✅ `cell_id`를 문자열로 변환

    # ✅ 3. 특정 셀의 유전자 발현량 데이터 가져오기
    cell_index = transcript_df.index
    if len(cell_index) == 0:
        print(f"⚠️ Cell ID {cell_id}의 transcript 데이터를 찾을 수 없음.")
        return
    
    cell_index = transcript_data.obs.index.get_loc(cell_index[0])
    gene_expression = transcript_data.X[cell_index, :].toarray().flatten()  # sparse matrix → array 변환
    gene_names = transcript_data.var.index.to_list()
    gene_expression_df = pd.DataFrame({"gene": gene_names, "expression": gene_expression})

    # ✅ 4.`gene_expression_df`의 `cell_id` 추가 및 문자열 변환
    gene_expression_df["cell_id"] = str(cell_id)

    # ✅ 5. Annotation 정보 가져오기
    cell_annotation_info = annotation_df[annotation_df["cell_id"] == cell_id]
    if cell_annotation_info.empty:
        print(f"⚠️ Cell ID {cell_id}의 Annotation 정보를 찾을 수 없음.")
        return

    # ✅ 6. 데이터 병합 (Transcript + Annotation + Gene Expression)
    merged_df = transcript_df.merge(cell_annotation_info, on="cell_id", how="left")
    merged_df = merged_df.merge(gene_expression_df, on="cell_id", how="left")  # ✅ `right_index=True` 제거

    # ✅ 7. 이미지 저장 (Matplotlib으로 저장)
    
    # ✅ 저장할 폴더 경로 설정
    save_path = f"/data0/paired_dataset/{organ}/he{upper_threshold}"
    os.makedirs(save_path, exist_ok=True)  # 폴더가 없으면 생성
    
    image_filename = f"{save_path}/{cell_id}.png"
    csv_filename = f"{save_path}cell_{cell_id}_transcripts_with_annotation.csv"
    
    # ✅ 8. 크롭된 이미지 저장 (Matplotlib)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(cropped_image.transpose(1, 2, 0))  # (C, Y, X) → (Y, X, C) 변환
    #ax.set_title("Cropped H&E Image (Aligned to Xenium Cell)")
    ax.axis("off")  # 축 제거
    
    # ✅ 9. 이미지 저장 (bbox_inches='tight'로 불필요한 여백 제거)
    fig.savefig(image_filename, bbox_inches="tight", pad_inches=0, dpi=300)
    plt.close(fig)  # 메모리 해제
    
    #print(f"✅ Cropped H&E Image saved: {image_filename}")
    # ✅ 10. CSV 저장
    #merged_df.to_csv(csv_filename, index=False)

    #print(f"✅ H&E Image saved: {image_filename}")
    #print(f"✅ Transcript + Annotation saved: {csv_filename}")
    return gene_expression_df[["gene", "expression", "cell_id"]]

In [37]:
def process_all_cells(pixel_size, affine_matrix_path, save_csv=True, save_adata=True):
    """중앙 80% cell_area 범위의 셀을 처리하고, AnnData 형식으로 저장하는 함수"""

    # ✅ 1. 필터링된 cell_id 리스트 생성
    filtered_cell_ids = set(filtered_cells["cell_id"])
    final_cell_set = filtered_cell_ids & set(cell_ids)
    final_cell = list(final_cell_set)

    # ✅ 2. 병렬 처리
    pool = Pool(num_cores)
    start_time = time.time()
    expression_dfs = pool.map(process_cell, final_cell)
    pool.close()
    pool.join()
    end_time = time.time()
    print(f"📦 자료 생성 시간 : {end_time - start_time:.4f}초")

    # ✅ 3. None 제거 (처리 실패 셀 제외)
    expression_dfs = [df for df in expression_dfs if df is not None]

    # ✅ 4. long-form DataFrame 병합
    all_expr_df = pd.concat(expression_dfs, axis=0)

    # ✅ 5. wide-form 변환 → Cell-by-Gene matrix
    expr_matrix = all_expr_df.pivot(index="cell_id", columns="gene", values="expression").fillna(0)

    # ✅ 6. AnnData 객체 생성
    adata = sc.AnnData(
        X=sparse.csr_matrix(expr_matrix.values),
        obs=pd.DataFrame(index=expr_matrix.index),
        var=pd.DataFrame(index=expr_matrix.columns)
    )

    # ✅ 7. annotation 정보 추가 (cell_id 기준 병합)
    adata.obs = adata.obs.merge(annotation_df.set_index("cell_id"), left_index=True, right_index=True, how="left")

    # ✅ 8. 그룹별 cell 수 출력
    group_counts = adata.obs["group"].value_counts()
    print("\n📊 ✅ Filtered 이후 Group별 Cell 개수:")
    print(group_counts)

    # ✅ annotation.csv 불러오기
    annotation_path = f"/data0/paired_dataset/{organ}/annotation.csv"
    anno_df = pd.read_csv(annotation_path).set_index("cell_id")

    # ✅ 9. 저장
    if save_csv:
        # ➤ group별 cell 개수 저장
        save_path = f"/data0/xenium/{organ}_cell_info_{upper_threshold}"
        os.makedirs(save_path, exist_ok=True)
        csv_filename_counts = f"{save_path}/filtered_group_counts.csv"
        group_counts.to_csv(csv_filename_counts, header=True)
        print(f"✅ 그룹별 cell 개수가 '{csv_filename_counts}'로 저장되었습니다.")

        # ➤ annotation 병합 + transpose 저장
        merged_expr_T = anno_df.join(expr_matrix, how="inner").T

        # ➤ 원본 shape
        total_genes = merged_expr_T.shape[0]
        total_cells = merged_expr_T.shape[1]

        # ✅ 1. 전부 0인 gene 제거
        nonzero_expr_T = merged_expr_T.loc[(merged_expr_T != 0).any(axis=1)]

        # ➤ 필터링 후 shape
        remaining_genes = nonzero_expr_T.shape[0]
        remaining_cells = nonzero_expr_T.shape[1]

        # ✅ 2. 저장
        csv_filename_expr_filtered = f"{save_path}/gene_by_cell_expression_matrix_with_annotation_non_zero_filtered.csv"
        nonzero_expr_T.to_csv(csv_filename_expr_filtered)
        print(f"✅ 0만 있는 유전자 제외한 matrix 저장 완료: {csv_filename_expr_filtered}")

        csv_filename_expr_annot = f"{save_path}/gene_by_cell_expression_matrix_with_annotation.csv"
        merged_expr_T.to_csv(csv_filename_expr_annot)
        print(f"✅ Annotation 포함된 matrix가 '{csv_filename_expr_annot}'로 저장되었습니다.")

        # ✅ 3. 제거된 gene / cell 개수 출력
        print(f"\n🧬 총 gene 수: {total_genes} → 필터 후: {remaining_genes}")
        print(f"🧬 제거된 gene 수: {total_genes - remaining_genes}")

        if total_cells == remaining_cells:
            print("✅ 모든 cell이 유지되었습니다.")
        else:
            print(f"⚠️ 제거된 cell 수: {total_cells - remaining_cells}")

    if save_adata:
        adata_filename = f"/data0/paired_dataset/{organ}/expression_he{upper_threshold}.h5ad"
        adata.write(adata_filename)
        print(f"✅ AnnData가 '{adata_filename}'로 저장되었습니다.")

    return adata

In [45]:
def cell_area_filter(cell_areas, upper_threshold):
    # ✅ 상위 10%와 하위 10%를 제외한 범위 계산
    lower_percentile = np.percentile(cell_areas, 10)
    upper_threshold = upper_threshold  # 상위 10% 대신 200 미만으로 제한
    #upper_threshold = 300  # 상위 10% 대신 300 미만으로 제한
        # ✅ 중앙 범위 내 cell_area 필터링
    filtered_cells = transcript_data.obs[
        (cell_areas >= lower_percentile) & (cell_areas < upper_threshold)
    ]
    
    # ✅ 중앙 80% 범위 내 최대 cell_area 찾기
    max_filtered_area_index = filtered_cells["cell_area"].idxmax()
    max_filtered_cell_id = filtered_cells.loc[max_filtered_area_index, "cell_id"]
    
    # ✅ 중앙 80% 범위 내 최대 cell_area 찾기
    min_filtered_area_index = filtered_cells["cell_area"].idxmin()
    min_filtered_cell_id = filtered_cells.loc[min_filtered_area_index, "cell_id"]
    
    print(f"🔍 중앙 80% 범위 내 가장 큰 cell_id: {max_filtered_cell_id}")
    print(f"🔹 중앙 80% 범위 내 최대 cell_area: {filtered_cells.loc[max_filtered_area_index, 'cell_area']}")
    print(f"🔍 중앙 80% 범위 내 가장 작은 cell_id: {min_filtered_cell_id}")
    print(f"🔹 중앙 80% 범위 내 최소 cell_area: {filtered_cells.loc[min_filtered_area_index, 'cell_area']}")
    # 예제 사용
    cell_boundaries = sdata.shapes["cell_boundaries"].copy()
    max_area_polygon = cell_boundaries.loc[max_filtered_cell_id, "geometry"]  # 가장 큰 polygon
    obb_width, obb_height = get_obb_dimensions(max_area_polygon)

    #print(f"🔹 OBB 크기: {obb_width} x {obb_height}")
    # μm 단위의 OBB 크기를 픽셀로 변환
    obb_width_px = obb_width / pixel_size
    obb_height_px = obb_height / pixel_size
    print(obb_width_px)
    print(obb_height_px)
    print(obb_width)
    print(obb_height)
    crop_size = int(obb_width_px)
    crop_size += crop_size % 2  # 홀수이면 +1 하여 짝수로 변환
    
    #print(f"🔹 OBB 크기 (픽셀 기준): {obb_width_px:.2f} x {obb_height_px:.2f}")
    print(f"🔹 선택된 Crop Size: {crop_size}")
    
    return filtered_cells, crop_size

In [35]:
def process_all_cells(pixel_size, affine_matrix_path, save_csv=True, save_adata=True):
    """중앙 80% cell_area 범위의 셀을 처리하고, AnnData 형식으로 저장하는 함수"""
    
    # ✅ 1. 필터링된 cell_id 리스트 생성
    filtered_cell_ids = set(filtered_cells["cell_id"])
    final_cell_set = filtered_cell_ids & set(cell_ids)
    final_cell = list(final_cell_set)

    # ✅ 2. 병렬 처리
    pool = Pool(num_cores)
    start_time = time.time()
    expression_dfs = pool.map(process_cell, final_cell)
    pool.close()
    pool.join()
    end_time = time.time()
    print(f"📦 자료 생성 시간 : {end_time - start_time:.4f}초")

    # ✅ 3. None 제거 (처리 실패 셀 제외)
    expression_dfs = [df for df in expression_dfs if df is not None]

    # ✅ 4. long-form DataFrame 병합
    all_expr_df = pd.concat(expression_dfs, axis=0)

    # ✅ 5. wide-form 변환 → Cell-by-Gene matrix
    expr_matrix = all_expr_df.pivot(index="cell_id", columns="gene", values="expression").fillna(0)

    # ✅ 6. AnnData 객체 생성
    adata = sc.AnnData(
        X=sparse.csr_matrix(expr_matrix.values),
        obs=pd.DataFrame(index=expr_matrix.index),
        var=pd.DataFrame(index=expr_matrix.columns)
    )

    # ✅ 7. annotation 정보 추가 (cell_id 기준 병합)
    adata.obs = adata.obs.merge(annotation_df.set_index("cell_id"), left_index=True, right_index=True, how="left")

    # ✅ 8. 그룹별 cell 수 출력
    group_counts = adata.obs["group"].value_counts()
    print("\n📊 ✅ Filtered 이후 Group별 Cell 개수:")
    print(group_counts)

    # annotation.csv 불러오기
    annotation_path = f"/data0/paired_dataset/{organ}/annotation.csv"
    anno_df = pd.read_csv(annotation_path).set_index("cell_id")

    # ✅ 9. 저장
    if save_csv:
        # ➤ group별 cell 개수 저장
        save_path = f"/data0/xenium/{organ}_cell_info_{upper_threshold}"
        os.makedirs(save_path, exist_ok=True)  # 폴더가 없으면 생성
        csv_filename_counts = f"{save_path}/filtered_group_counts.csv"
        group_counts.to_csv(csv_filename_counts, header=True)
        print(f"✅ 그룹별 cell 개수가 '{csv_filename_counts}'로 저장되었습니다.")
        
        # ➤ expression matrix도 CSV로 저장
        #csv_filename_expr = f"/data0/xenium/{organ}_cell_info_{upper_threshold}/cell_by_gene_expression_matrix.csv"
        #csv_filename_expr = f"{save_path}/gene_by_cell_expression_matrix.csv"
        #expr_matrix.to_csv(csv_filename_expr)
        #expr_matrix.T.to_csv(csv_filename_expr)  # 👈 transpose 추가
        #print(f"✅ Cell-by-gene expression matrix가 '{csv_filename_expr}'로 저장되었습니다.")
        
        # ➤ annotation 병합 + transpose 저장
        
        # ➤ 0. annotation 병합
        merged_expr_T = anno_df.join(expr_matrix, how="inner").T
        
        # ✅ 1. 발현값이 모든 셀에서 0인 gene 제거
        nonzero_expr_T = merged_expr_T.loc[(merged_expr_T != 0).any(axis=1)]
        
        # ✅ 2. CSV로 저장
        csv_filename_expr_filtered = f"{save_path}/gene_by_cell_expression_matrix_with_annotation_non_zero_filtered.csv"
        nonzero_expr_T.to_csv(csv_filename_expr_filtered)
        print(f"✅ 0만 있는 유전자 제외한 matrix 저장 완료: {csv_filename_expr_filtered}")

        csv_filename_expr_annot = f"{save_path}/gene_by_cell_expression_matrix_with_annotation.csv"
        merged_expr_T.to_csv(csv_filename_expr_annot)
        print(f"✅ Annotation 포함된 matrix가 '{csv_filename_expr_annot}'로 저장되었습니다.")

    if save_adata:
        adata_filename = f"/data0/paired_dataset/{organ}/expression_he{upper_threshold}.h5ad"
        adata.write(adata_filename)
        print(f"✅ AnnData가 '{adata_filename}'로 저장되었습니다.")

    return adata

# Xenium data -LUNG

In [46]:
xenium_path = "./xenium_prime_link/Human_Lung_Cancer/data.zarr"

In [47]:
sdata = sd.read_zarr(xenium_path)

/home/david/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:183: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [48]:
sdata

SpatialData object, with associated Zarr store: /data0/xenium_prime/Human_Lung_Cancer/data.zarr
├── Images
│     ├── 'he_image': DataTree[cyx] (3, 43270, 26720), (3, 21635, 13360), (3, 10817, 6680), (3, 5408, 3340), (3, 2704, 1670)
│     └── 'morphology_focus': DataTree[cyx] (5, 37348, 54086), (5, 18674, 27043), (5, 9337, 13521), (5, 4668, 6760), (5, 2334, 3380)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (37348, 54086), (18674, 27043), (9337, 13521), (4668, 6760), (2334, 3380)
│     └── 'nucleus_labels': DataTree[yx] (37348, 54086), (18674, 27043), (9337, 13521), (4668, 6760), (2334, 3380)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (278328, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (278328, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (275207, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (278328, 5001)
with coordinate systems:
   

# annotation df generation

In [18]:
# ✅ Annotation 정보가 포함된 CSV 파일 로드
annotation_file = "annotation/merged_output.csv"  # 🛠 파일 경로 확인 필요
annotation_df = pd.read_csv(annotation_file)

In [19]:
# cell_id를 index로 설정하고 group만 남기기
annotation_clean = annotation_df.set_index("cell_id")[["group"]]

In [20]:
# 파일 저장 경로 구성
dir_path = f"/data0/paired_dataset/{organ}"
file_path = os.path.join(dir_path, "annotation.csv")

# 디렉토리 만들기
os.makedirs(dir_path, exist_ok=True)

# 저장
annotation_clean.to_csv(file_path)

In [21]:
#df = pd.read_csv(file_path, index_col=0)  # 다시 불러올 때 index로 인식

# Cell Extraction -LUNG

In [22]:
# ✅ `cell_id`를 문자열로 변환 (Annotation 데이터)
annotation_df["cell_id"] = annotation_df["cell_id"].astype(str)

In [23]:
# ✅ Transcript 데이터 가져오기
transcript_data = sdata.tables["table"]

In [24]:
transcript_data.obs

,cell_id,transcript_counts,control_probe_counts,genomic_control_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,cell_area,nucleus_area,nucleus_count,segmentation_method,region,z_level,cell_labels
0,aaaaadnb-1,226,0,0,0,0,0,226,44.659533,21.223438,1.0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,cell_circles,0.0,1
1,aaaabalp-1,108,0,0,0,0,0,108,32.196407,9.889219,1.0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,cell_circles,0.0,2
2,aaaadfei-1,265,0,0,0,0,3,268,47.504377,25.739063,1.0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,cell_circles,0.0,3
3,aaaadjia-1,328,0,0,0,0,1,329,37.253908,20.636407,1.0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,cell_circles,0.0,4
4,aaaafglb-1,337,0,0,0,0,0,337,52.155471,27.274376,1.0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,cell_circles,0.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278323,oikioede-1,76,0,0,0,0,0,76,41.001876,41.001876,1.0,Segmented by nucleus expansion of 5.0µm,cell_circles,7.0,278324
278324,oikiokmm-1,12,0,0,0,0,0,12,26.416407,26.416407,1.0,Segmented by nucleus expansion of 5.0µm,cell_circles,6.0,278325
278325,oikjekcf-1,73,0,0,0,0,1,74,37.976408,27.545313,1.0,Segmented by nucleus expansion of 5.0µm,cell_circles,6.0,278326
278326,oikjghfi-1,9,0,0,0,0,0,9,26.326095,7.857188,1.0,Segmented by nucleus expansion of 5.0µm,cell_circles,5.0,278327


In [25]:
# ✅ H&E 이미지 가져오기
he_image = sdata.images["he_image"]["scale0"]
he_image_array = he_image["image"].values  # Xarray → NumPy 변환
image_channels, image_height, image_width = he_image_array.shape  # (C, Y, X)

In [26]:
# ✅ 모든 `cell_id` 리스트 가져오기 (annotation 파일 기반)
cell_ids = annotation_df["cell_id"].unique()

In [27]:
# ✅ Transcript 데이터에서 "cell_area" 값 가져오기
cell_areas = transcript_data.obs["cell_area"]

In [28]:
affine_matrix = load_affine_matrix(affine_matrix_path)

In [49]:
# ✅ 2. 중앙 80% 범위의 cell_id 필터링 및 crop_size 설정
filtered_cells, crop_size = cell_area_filter(transcript_data.obs["cell_area"], upper_threshold)

🔍 중앙 80% 범위 내 가장 큰 cell_id: amggapia-1
🔹 중앙 80% 범위 내 최대 cell_area: 199.99703850969672
🔍 중앙 80% 범위 내 가장 작은 cell_id: abbghhei-1
🔹 중앙 80% 범위 내 최소 cell_area: 24.203750878572464
117.64705882352942
117.64705882352942
25
25
🔹 선택된 Crop Size: 118


In [29]:
# ✅ 2. 중앙 80% 범위의 cell_id 필터링 및 crop_size 설정
filtered_cells, crop_size = cell_area_filter(transcript_data.obs["cell_area"], upper_threshold)

🔍 중앙 80% 범위 내 가장 큰 cell_id: amggapia-1
🔹 중앙 80% 범위 내 최대 cell_area: 199.99703850969672
🔍 중앙 80% 범위 내 가장 작은 cell_id: abbghhei-1
🔹 중앙 80% 범위 내 최소 cell_area: 24.203750878572464
🔹 선택된 Crop Size: 118


In [38]:
process_all_cells(pixel_size, affine_matrix_path)

📦 자료 생성 시간 : 108.4633초

📊 ✅ Filtered 이후 Group별 Cell 개수:
group
Stromal_cell        5068
Tumor_cell_LUAD     4704
Lymphocyte          4668
Pericyte            3912
Endothelial_cell    1158
Name: count, dtype: int64
✅ 그룹별 cell 개수가 '/data0/xenium/10x_5k_lung_cell_info_200/filtered_group_counts.csv'로 저장되었습니다.
✅ 0만 있는 유전자 제외한 matrix 저장 완료: /data0/xenium/10x_5k_lung_cell_info_200/gene_by_cell_expression_matrix_with_annotation_non_zero_filtered.csv
✅ Annotation 포함된 matrix가 '/data0/xenium/10x_5k_lung_cell_info_200/gene_by_cell_expression_matrix_with_annotation.csv'로 저장되었습니다.

🧬 총 gene 수: 5002 → 필터 후: 4998
🧬 제거된 gene 수: 4
✅ 모든 cell이 유지되었습니다.
✅ AnnData가 '/data0/xenium/paired_dataset/10x_5k_lung/expression_he200.h5ad'로 저장되었습니다.


AnnData object with n_obs × n_vars = 19510 × 5001
    obs: 'group'

In [ ]:
# ✅ 2. 중앙 80% 범위의 cell_id 필터링 및 crop_size 설정
filtered_cells, crop_size = cell_area_filter(transcript_data.obs["cell_area"], upper_threshold)

In [ ]:
process_all_cells(pixel_size, affine_matrix_path)

In [ ]:
exp_mat = pd.read_csv('./xenium_link/lung_cell_info_200/cell_by_gene_expression_matrix.csv',index_col=0)

In [ ]:
gene_by_cell = exp_mat.T

In [ ]:
gene_by_cell

In [ ]:
exp_mat.head()

In [ ]:
process_all_cells(pixel_size, affine_matrix_path)

In [ ]:
adata = sc.read_h5ad("/data0/xenium/cell_by_gene_expression_matrix.h5ad")

In [ ]:
df = pd.DataFrame(
    adata.X.toarray(),
    index=adata.obs_names,
    columns=adata.var_names
)
df.head()

In [ ]:
sdata.tables["table"]

In [ ]:
mp.cpu_count()

In [ ]:
transcript_data.obs["cell_area"]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Violin plot 생성
plt.figure(figsize=(8, 5))
sns.violinplot(y=transcript_data.obs["cell_area"])
plt.title("Violin Plot of Cell Area")
plt.ylabel("Cell Area")
plt.grid(True)

# 그래프 출력
plt.show()

In [ ]:
num_cells_above_300 = (transcript_data.obs["cell_area"] >= 300).sum()
print(num_cells_above_300)

In [ ]:
# cell_area의 상위 10% 기준값 계산
threshold = transcript_data.obs["cell_area"].quantile(0.90)

# 상위 10%에 해당하는 cell 수 계산
num_top_10_percent = (transcript_data.obs["cell_area"] >= threshold).sum()

# 결과 출력
print(num_top_10_percent)


In [ ]:
# ✅ 3. 필터링된 cell_id 리스트 생성
    filtered_cell_ids = set(filtered_cells["cell_id"])

    # ✅ 4. 중앙 80% 범위 내 `cell_id`만 처리
    processed_cells = []

    final_cell_set = filtered_cell_ids & set(cell_ids)
    final_cell = list(final_cell_set)                     

In [ ]:
# ✅ 1. 가장 큰 셀의 polygon 가져오기
cell_boundaries = sdata.shapes["cell_boundaries"].copy()
max_area_polygon = cell_boundaries.loc[max_filtered_cell_id, "geometry"]  # 가장 큰 polygon

# ✅ 2. Polygon을 픽셀 단위로 변환
max_area_polygon_px = convert_polyum_to_px(max_area_polygon, pixel_size)  # μm → 픽셀 변환

# ✅ 3. 변환된 Polygon으로 OBB 크기 구하기
obb_width_px, obb_height_px = get_obb_dimensions(max_area_polygon_px)  # 이미 픽셀 단위

# ✅ 4. 최종 crop_size 설정
crop_size = int(obb_width_px)

print(f"🔹 OBB 크기 (픽셀 기준): {obb_width_px:.2f} x {obb_height_px:.2f}")
print(f"🔹 선택된 Crop Size: {crop_size}")
